# Avaliação normal do novo dataset (wrapper fino)

Este notebook **não implementa** o protocolo. Ele chama a CLI do pacote
`new_data_eval`, a mesma usada em `PYTHONPATH=code python -m new_data_eval normal`.
A lógica de features, máscaras, RePO/RePO+, thresholds e métricas vive apenas
no pacote (`code/new_data_eval/`). Veja `docs/NEW_DATASET.md`.

Para o protocolo completo, use `MAX_SAMPLES = None` e `N_MASKS = 100`; os
valores atuais são apenas para inspeção/smoke. Os resultados são gravados em
`results/new_data/notebook_smoke/normal/`, isolados dos resultados do
protocolo completo em `results/new_data/normal/`.

In [ ]:
from pathlib import Path
import sys


def find_repo_root(start):
    """Return the repository root that contains ``code/new_data_eval``."""
    for candidate in (start, *start.parents):
        if (candidate / "code" / "new_data_eval").is_dir():
            return candidate
    raise RuntimeError("Could not find the repository root")


REPO_ROOT = find_repo_root(Path.cwd().resolve())
sys.path.insert(0, str(REPO_ROOT / "code"))

from new_data_eval import cli

## Parâmetros de inspeção

`MAX_SAMPLES` limita quantas linhas válidas do teste são lidas e `N_MASKS`
reduz o número de máscaras por amostra. Ambos existem para smoke tests; não
alteram o protocolo quando ficam em `None`/`100`.

In [ ]:
DATA_DIR = REPO_ROOT / "data" / "new_data"
MODEL_DIR = REPO_ROOT / "models" / "new_data_model"
RESULTS_DIR = Path(cli.DEFAULT_RESULTS_DIR) / "notebook_smoke"

# Smoke defaults: small and quick.  For the full protocol use
# MAX_SAMPLES = None and N_MASKS = 100.
MAX_SAMPLES = 20
N_MASKS = 5
BATCH_SIZE = 2
SEED = 2019

## Validar dados, metadados e modelo

In [ ]:
validate_args = [
    "validate",
    "--data-dir", str(DATA_DIR),
    "--model-dir", str(MODEL_DIR),
    "--seed", str(SEED),
]
if cli.main(validate_args) != 0:
    raise RuntimeError("dataset/model validation failed")

## Rodar a avaliação normal (RePO e RePO+)

In [ ]:
normal_args = [
    "normal",
    "--data-dir", str(DATA_DIR),
    "--model-dir", str(MODEL_DIR),
    "--results-dir", str(RESULTS_DIR),
    "--seed", str(SEED),
    "--batch-size", str(BATCH_SIZE),
    "--n-masks", str(N_MASKS),
]
if MAX_SAMPLES is not None:
    normal_args += ["--max-samples", str(MAX_SAMPLES)]
if cli.main(normal_args) != 0:
    raise RuntimeError("normal evaluation failed")

## Inspecionar os JSONs gravados

In [ ]:
import json

for mode in ("repo", "repo_plus"):
    result_file = RESULTS_DIR / "normal" / f"{mode}.json"
    payload = json.loads(result_file.read_text(encoding="utf-8"))
    print(mode, "->", result_file)
    print("thresholds:", payload["thresholds"])
    print("fpr:", payload["fpr"])
    print("average_tpr:", payload["average_tpr"])
    for label, rate in payload["class_rates"]["at_fpr_0.01"].items():
        print(f"  {label}: TPR={rate['tpr']:.4f} (n={rate['n']})")